# Семинар 4. Свёртки и свёрточные сети

Основной трек: Chapter 5 D. V. Godoy. Начинаем с представления изображений из
Chapter 4, затем проходим от одного окна фильтра до классификатора.
Сравнение MLP и CNN на MNIST является отдельным экспериментом курса.
Используем организацию обучения из третьего семинара, без новой обёртки над моделью.


## Содержание

1. [Изображение как тензор](#section-1)
2. [Фильтр и одно окно](#section-2)
3. [Свёртка в PyTorch](#section-3)
4. [Размер выхода, stride и padding](#section-4)
5. [Фильтры на изображении](#section-5)
6. [Pooling и Flatten](#section-6)
7. [Каналы и свёртка 1x1](#section-7)
8. [Собираем CNN](#section-8)
9. [Протокол сравнения MLP и CNN](#section-9)
10. [Обучение и валидация](#section-10)
11. [Фильтры и карты признаков](#section-11)
12. [Проверяем сдвиги](#section-12)
13. [Задания](#section-13)
14. [Итоги](#section-14)


In [ ]:
from pathlib import Path
from runpy import run_path

# При согласованной проверке Colab замените main на SHA коммита.
course_revision = 'main'
try:
    import google.colab
except ModuleNotFoundError:
    repository_root = next(
        (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (p / 'config.py').is_file() and (p / 'plots' / 'seminar04.py').is_file()), None)
    if repository_root is None:
        raise RuntimeError('Откройте notebook из локальной копии 2026_ML3.')
else:
    repository_root = Path.cwd().resolve()
    from urllib.request import urlopen
    url = f'https://raw.githubusercontent.com/SergeyMalashenko/2026_ML3/{course_revision}/config.py'
    with urlopen(url, timeout=30) as response:
        config_source = response.read()
    compile(config_source, 'config.py', 'exec')
    (repository_root / 'config.py').write_bytes(config_source)
course_config = run_path(str(repository_root / 'config.py'))
seminar_plots = course_config['config_seminar04'](branch=course_revision)
select_device = course_config['select_device']


In [ ]:
import os
import random
from time import perf_counter
import numpy as np
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision.datasets import MNIST
import matplotlib.pyplot as plt

seminar_plots.configure_plots()
device = select_device()  # Явная проверка на CPU: select_device('cpu').
data_root = Path(os.environ.get('AI_MASTERS_DATA', Path.home() / '.cache' / 'ai-masters-ml3'))
print('Устройство:', device)


<a id="section-1"></a>

## 1. Изображение как тензор

В PyTorch изображение обычно имеет форму `(C, H, W)`, а батч изображений
имеет форму `(N, C, H, W)`. В MNIST один канал и десять классов.
`Flatten` меняет форму, но не удаляет пиксели: при известной исходной форме преобразование обратимо.

**Вопрос:** что полносвязный слой получает после Flatten? Заложена ли в нём близость соседних пикселей?


In [ ]:
mnist = MNIST(root=data_root, train=True, download=True)
example_images = mnist.data[:8].float() / 255
seminar_plots.show_images(example_images, [f'Класс {v}' for v in mnist.targets[:8]]);
print('Один объект:', (1, *mnist.data.shape[1:]))
print('После Flatten:', nn.Flatten()(example_images[:, None]).shape)


<a id="section-2"></a>

## 2. Фильтр и одно окно

Как у Годоя, сначала рассмотрим маленькое изображение и одно ядро.
Для каждой позиции берём окно, перемножаем его элементы с весами ядра и складываем.
Одни и те же веса используются во всех позициях.

В `Conv2d` ядро не переворачивается: математически это взаимная корреляция.
В нейросетях эту операцию традиционно называют свёрткой.

**Перед запуском:** чему равен первый результат? А результат после сдвига окна вправо на один пиксель?


<details>
<summary>Дополнение: кросс-корреляция и математическая свёртка</summary>

Это самостоятельная справка, не дополнительный шаг основного эксперимента.
Рассмотрим вещественные сигналы на целочисленной решётке. Для конечных массивов
сначала фиксируем продолжение за границы и положение начала координат.

**Определения.** Математическая свёртка:

$$
(I*K)_{i,j}=\sum_m\sum_n I_{i-m,j-n}K_{m,n}.
$$

Кросс-корреляция (взаимная корреляция, без комплексного сопряжения для вещественных данных):

$$
(I\star K)_{i,j}=\sum_m\sum_n I_{i+m,j+n}K_{m,n}.
$$

**Переход через переворот ядра.** Определим

$$K'_{m,n}=K_{-m,-n}.$$

Подставим это ядро в свёртку и заменим индексы: $u=-m$, $v=-n$.

$$
\begin{aligned}
(I*K')_{i,j}
&=\sum_m\sum_n I_{i-m,j-n}K_{-m,-n}\\
&=\sum_u\sum_v I_{i+u,j+v}K_{u,v}\\
&=(I\star K)_{i,j}.
\end{aligned}
$$

**Конечное окно без padding, stride 1.** Для массива `[a,b,c,d]` и ядра `[x,y,z]`
два положения окна дают следующие результаты.

Кросс-корреляция:

$$[ax+by+cz,\; bx+cy+dz].$$

Свёртка, записанная по тем же положениям окна с перевёрнутым ядром:

$$[az+by+cx,\; bz+cy+dx].$$

В определении на решётке это valid-значения свёртки с индексами 2 и 3;
при записи короткого выходного массива мы перенумеровали их с нуля.
Это не весь результат full-свёртки, включающий частичные перекрытия.

Например, для `[1,2,4,8]` и `[1,0,-1]` корреляция даёт `[-3,-6]`,
а свёртка с переворотом ядра даёт `[3,6]`. Для симметричного ядра ответы совпадут.

**В PyTorch.** `F.conv1d` и `F.conv2d` вычисляют взаимную корреляцию.
Для математической свёртки в таком оконном соглашении передайте
`kernel.flip(-1)` в 1D или `kernel.flip(-2, -1)` в 2D.
Одинаковыми должны оставаться правила padding, stride и выбор выходных позиций.
При свободно обучаемом ядре переворот — взаимно однозначная перепараметризация:
класс доступных преобразований тот же. Для заданного вручную несимметричного
фильтра конкретный ответ меняется.

Основа: разбор семинара 4 (2025), с исправленной индексацией и уточнением valid-режима.
Документация: [Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html).

</details>


In [ ]:
single = np.array(
    [[[[5, 0, 8, 7, 8, 1],
       [1, 9, 5, 0, 7, 7],
       [6, 0, 2, 4, 6, 6],
       [9, 7, 6, 6, 8, 4],
       [8, 3, 8, 5, 1, 3],
       [7, 2, 7, 0, 1, 0]]]]
)
single.shape


In [ ]:
identity = np.array(
    [[[[0, 0, 0],
       [0, 1, 0],
       [0, 0, 0]]]]
)
identity.shape


In [ ]:
seminar_plots.plot_window(single, identity);


In [ ]:
region = single[:, :, 0:3, 0:3]
filtered_region = region * identity
total = filtered_region.sum()
total


In [ ]:
new_region = single[:, :, 0:3, (0+1):(3+1)]


In [ ]:
new_filtered_region = new_region * identity
new_total = new_filtered_region.sum()
new_total


In [ ]:
seminar_plots.plot_window(single, identity, col=1);


<a id="section-3"></a>

## 3. Свёртка в PyTorch

`F.conv2d` применяет переданное ядро. `nn.Conv2d` хранит обучаемые
параметры и вызывает операцию с ними. Это различие операции и слоя с состоянием.

**Вопрос:** почему произвольный `nn.Conv2d` пока не обязан выдавать результат нашего ручного фильтра?


In [ ]:
image = torch.as_tensor(single).float()
kernel_identity = torch.as_tensor(identity).float()


In [ ]:
convolved = F.conv2d(image, kernel_identity, stride=1)
convolved


In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, stride=1)
conv(image)


In [ ]:
with torch.no_grad():
    conv.weight[0] = kernel_identity
    conv.bias[0] = 0


In [ ]:
conv(image)


In [ ]:
torch.testing.assert_close(conv(image), convolved)
print('Операция и слой с одинаковыми весами дают одинаковый результат.')


<a id="section-4"></a>

## 4. Размер выхода, stride и padding

Для каждой пространственной оси размер выхода равен

$$H_{out}=\left\lfloor\frac{H_{in}+2p-d(k-1)-1}{s}+1\right\rfloor.$$

Здесь $k$ — размер ядра, $p$ — padding с каждой стороны, $s$ — stride,
$d$ — dilation. В основных примерах $d=1$.
Окно должно целиком помещаться в дополненный вход; неполное последнее окно не вычисляется.

**Вопрос:** сколько положений у окна размера 3 на оси длины 6 при stride 2 и без padding?
А на оси длины 7? Сначала перечислите стартовые индексы, затем примените формулу.

Интерактивная схема: [Convolution Visualizer](https://ezyang.github.io/convolution-visualizer/). Меняйте размер ядра, padding, stride и dilation и сопоставляйте размер выхода с формулой.


In [ ]:
convolved_stride2 = F.conv2d(image, kernel_identity, stride=2)
convolved_stride2


In [ ]:
constant_padder = nn.ConstantPad2d(padding=1, value=0)
constant_padder(image)


In [ ]:
replication_padder = nn.ReplicationPad2d(padding=1)
replication_padder(image)


In [ ]:
reflection_padder = nn.ReflectionPad2d(padding=1)
reflection_padder(image)


In [ ]:
def output_size(length, kernel, padding=0, stride=1, dilation=1):
    return (length + 2*padding - dilation*(kernel-1) - 1)//stride + 1

settings = [(0, 1), (1, 1), (0, 2), (1, 2)]
fixed_results = [F.conv2d(image, kernel_identity, padding=p, stride=s) for p, s in settings]
for (p, s), result in zip(settings, fixed_results):
    expected = output_size(6, 3, p, s)
    assert result.shape[-2:] == (expected, expected)
seminar_plots.show_images(fixed_results,
    [f'p={p}, s={s}: {tuple(y.shape[-2:])}' for (p,s),y in zip(settings, fixed_results)]);


<a id="section-5"></a>

## 5. Фильтры на изображении

Теперь применим фиксированные ядра к изображению прошлого года.
Усреднение сглаживает изображение. Фильтры Собеля выделяют изменения яркости
по двум осям; их отклики являются разными компонентами, а не двумя оценками одного числа.

**Вопрос:** может ли сумма двух больших по модулю компонент оказаться нулевой?
Будет ли это означать, что границы нет?


In [ ]:
photo = Image.open(seminar_plots.image_path()).convert('RGB')
photo.thumbnail((320, 240))
rgb = torch.from_numpy(np.array(photo).copy()).permute(2, 0, 1).float() / 255
gray = torch.from_numpy(np.array(photo.convert('L')).copy()).float()[None, None] / 255
box = torch.ones(1, 1, 5, 5) / 25
blurred = F.conv2d(F.pad(gray, (2, 2, 2, 2), mode='reflect'), box)
sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
sobel = torch.stack([sobel_x, sobel_x.T])[:, None]
components = F.conv2d(F.pad(gray, (1, 1, 1, 1), mode='reflect'), sobel)
gradient_magnitude = components.square().sum(dim=1, keepdim=True).sqrt()
seminar_plots.show_images([gray, blurred, gradient_magnitude],
                         ['Яркость', 'Усреднение 5x5', 'Длина градиента Собеля']);
seminar_plots.show_images([components[:, :1], components[:, 1:]],
                         ['Изменение по горизонтали', 'Изменение по вертикали'], signed=True);


<a id="section-6"></a>

## 6. Pooling и Flatten

У Годоя следующий шаг — уменьшить пространственный размер, затем перейти к вектору.
Pooling выполняет фиксированную операцию внутри окна; у MaxPool и AvgPool нет обучаемых весов.
`Flatten` сохраняет порядок элементов. В классификаторе мы можем либо сохранить позиции
через Flatten, либо усреднить каждую карту признаков по пространству.

**Вопрос:** сколько чисел останется после MaxPool 2x2 со stride 2 из карты 6x6?
Что именно потеряет pooling, а что не потеряет Flatten?


In [ ]:
edge = np.array(
    [[[[0, 1, 0],
       [1, -4, 1],
       [0, 1, 0]]]]
)
kernel_edge = torch.as_tensor(edge).float()
kernel_edge.shape


In [ ]:
padded = F.pad(image, (1, 1, 1, 1), mode='constant', value=0)
conv_padded = F.conv2d(padded, kernel_edge, stride=1)


In [ ]:
pooled = F.max_pool2d(conv_padded, kernel_size=2)
pooled


In [ ]:
flattened = nn.Flatten()(pooled)
flattened


In [ ]:
averaged = F.avg_pool2d(image, kernel_size=2, stride=2)
box_stride = F.conv2d(image, torch.ones(1, 1, 2, 2)/4, stride=2)
torch.testing.assert_close(averaged, box_stride)
seminar_plots.show_images([image, F.max_pool2d(image, 2), averaged],
                         ['Вход 6x6', 'Максимум 3x3', 'Среднее 3x3']);


<a id="section-7"></a>

## 7. Каналы и свёртка 1x1

У обычной свёртки с `groups=1` каждый выходной канал суммирует отклики
по всем входным каналам. Форма весов: `(C_out, C_in, kH, kW)`.

$$P=C_{out}(C_{in}k_Hk_W+1)$$

Единица соответствует bias каждого выходного канала; при `bias=False` её нет.
Число весов не зависит от высоты и ширины изображения.

Свёртка 1x1 смешивает каналы в каждой позиции, но не соседние пространственные позиции.
**Вопрос:** можно ли одним слоем 1x1 поменять местами красный и синий каналы, сохранив зелёный?


In [ ]:
channel_layer = nn.Conv2d(3, 4, kernel_size=3, padding=1)
print('Веса:', tuple(channel_layer.weight.shape))
print('Параметры:', sum(p.numel() for p in channel_layer.parameters()))
print('Выход:', tuple(channel_layer(rgb[None]).shape))

permutator = nn.Conv2d(3, 3, 1, bias=False)
with torch.no_grad():
    permutator.weight.copy_(torch.eye(3).flip(0).view(3, 3, 1, 1))
reordered = permutator(rgb[None])
torch.testing.assert_close(reordered[0], rgb.flip(0))
seminar_plots.show_images([rgb, reordered[0]], ['RGB', 'Поменяли красный и синий']);


<a id="section-8"></a>

## 8. Собираем CNN

Структура Годоя: извлечение признаков → уменьшение пространственного размера → классификатор.
Ниже вариант для MNIST из прошлого года, но без BatchNorm: это убирает ещё одно
различие со сравниваемой MLP. После трёх свёрток усредняем каждую карту признаков.
Последний слой выдаёт десять **логитов**, а не вероятности.

**Перед запуском:** восстановите формы после двух MaxPool и перед Linear.
Почему перед CrossEntropyLoss не добавляем Softmax?


In [ ]:
def make_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(8, 32, 3, padding=1), nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10),
    )

def make_mlp():
    return nn.Sequential(
        nn.Flatten(), nn.Linear(28*28, 26), nn.ReLU(),
        nn.Linear(26, 26), nn.ReLU(), nn.Linear(26, 10),
    )

model_cnn = make_cnn().eval()
with torch.no_grad():
    features = torch.zeros(2, 1, 28, 28)
    for layer in model_cnn:
        features = layer(features)
        print(type(layer).__name__, tuple(features.shape))


<a id="section-9"></a>

## 9. Протокол сравнения MLP и CNN

Сравниваем не «CNN вообще лучше MLP», а две конкретные архитектуры
при близком бюджете параметров. У них всё ещё разные глубина, связность и агрегация.
Равенство числа параметров не означает равенство FLOPs или одинаково удачные гиперпараметры.

Протокол фиксируем **до просмотра результатов**:
- 12 000 обучающих и 3 000 валидационных изображений из стандартного MNIST train;
- фиксированное случайное разбиение; официальный test здесь не используем;
- нормировка `(x/255 - 0.5)/0.5`, без аугментаций;
- Adam, lr=0.001, batch size 128, 5 эпох; одинаковый порядок батчей в паре;
- три seed: 17, 29, 43; финальные веса без выбора лучшей эпохи;
- train/validation метрики вычисляем в eval при одних и тех же весах после эпохи.

Остальные изображения не используются ради времени занятия. Это учебный эксперимент
на подвыборке, не результат на полном MNIST. Не меняйте протокол, чтобы добиться победы одной модели.
**Вопрос:** что контролирует равное число параметров, а чего оно не контролирует?


<details>
<summary>Дополнение: когда Dropout связан с L₂-регуляризацией</summary>

Это отдельный вывод для линейной регрессии с квадратичной потерей.
Он не утверждает эквивалентность Dropout и L₂ для произвольной нейросети.
В сравнении MLP/CNN выше Dropout не используется; протокол не меняем.

**Условия и обозначения.** Пусть $\hat y=b+\sum_{j=1}^d w_jx_j$,
а потеря одного объекта $\ell(y,\hat y)=\frac12(\hat y-y)^2$.
Маски $A_j\sim\operatorname{Bernoulli}(q)$ независимы;
$q\in(0,1]$ — вероятность **сохранения** признака, $p=1-q$ — вероятность
его зануления (`p` в `nn.Dropout`). Bias не маскируется.

**1. Сначала Dropout без компенсации масштаба.** Обозначим

$$S=\sum_j w_jA_jx_j,\qquad J=\mathbb E_A\!\left[\frac12(b+S-y)^2\right].$$

Здесь $x,y,w,b$ фиксированы: усредняем только по случайным маскам.

**2. Среднее и квадрат суммы.** Для каждой маски
$\mathbb E[A_j]=q$, $A_j^2=A_j$, а для разных индексов
$\mathbb E[A_jA_k]=q^2$ благодаря независимости. Поэтому

$$\mathbb E[S]=q\sum_j w_jx_j,$$

$$
\mathbb E[S^2]
=q\sum_j w_j^2x_j^2+q^2\sum_{j\ne k}w_jw_kx_jx_k.
$$

С другой стороны,

$$
(\mathbb E[S])^2
=q^2\sum_jw_j^2x_j^2+q^2\sum_{j\ne k}w_jw_kx_jx_k.
$$

Вычитая, получаем

$$\operatorname{Var}(S)=q(1-q)\sum_jw_j^2x_j^2.$$

**3. Ожидаемая потеря.** После раскрытия квадрата

$$
J=\frac12\left((b-y)^2+2(b-y)\mathbb E[S]+\mathbb E[S^2]\right).
$$

Подставим $\mathbb E[S^2]=(\mathbb E[S])^2+\operatorname{Var}(S)$:

$$
J=\frac12\left(b+q\sum_jw_jx_j-y\right)^2
+\frac{q(1-q)}2\sum_jw_j^2x_j^2.
$$

Это потеря **масштабированной модели** плюс квадратичный штраф,
взвешенный квадратами признаков. Bias в этом штрафе отсутствует.

**4. Inverted Dropout, как в PyTorch.** При обучении используется
$\tilde x_j=A_jx_j/q$. Тогда

$$\mathbb E[\tilde x_j]=x_j,\qquad
\operatorname{Var}(\tilde x_j)=\frac{1-q}{q}x_j^2,$$

$$
J_{\mathrm{inv}}
=\frac12\left(b+\sum_jw_jx_j-y\right)^2
+\frac{1-q}{2q}\sum_jw_j^2x_j^2.
$$

При `eval()` Dropout не изменяет вход. Для данного вывода важно $p<1$:
при $p=1$ деление на $q=0$ неприменимо.

**5. Когда это обычная L₂-регуляризация?** Усредним потери по $n$ объектам:

$$
L_{\mathrm{inv}}
=\frac1{2n}\sum_{i=1}^n\left(b+\sum_jw_jx_{ij}-y_i\right)^2
+\frac{1-q}{2q}\sum_jw_j^2\left(\frac1n\sum_i x_{ij}^2\right).
$$

Если для каждого признака эмпирический второй момент
$\frac1n\sum_i x_{ij}^2=1$, штраф равен

$$\frac\lambda2\lVert w\rVert_2^2,\qquad
\lambda=\frac{1-q}{q}=\frac{p}{1-p}.$$

Достаточно центрировать признаки и нормировать дисперсию с делителем $n$;
некоррелированность признаков для этого не требуется.
Без единичных вторых моментов штраф остаётся диагональным, но не изотропным.
Для Dropout **без** компенсации вместо этого получается $\lambda=q(1-q)$,
но вместе с масштабированной моделью из шага 3. Смешивать два соглашения нельзя.

Вывод использует половину MSE. Для обычного `nn.MSELoss()` без коэффициента 1/2
вся правая часть удваивается; коэффициент штрафа надо сравнивать в одной нормировке.
Равенство относится к ожидаемой цели, а не к каждому случайному шагу обучения.
Для нелинейной сети и cross-entropy данное доказательство не работает;
равенства с `weight_decay` произвольного оптимизатора оно также не устанавливает.

Основа: отдельная заметка курса о линейной модели; соглашение PyTorch сверено по
[документации Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html).

</details>


In [ ]:
split_generator = torch.Generator().manual_seed(2026)
indices = torch.randperm(len(mnist), generator=split_generator)
train_indices, val_indices = indices[:12000], indices[12000:15000]
x_train = (mnist.data[train_indices].float()[:, None]/255 - .5)/.5
y_train = mnist.targets[train_indices].clone()
x_val = (mnist.data[val_indices].float()[:, None]/255 - .5)/.5
y_val = mnist.targets[val_indices].clone()
train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)
assert not set(train_indices.tolist()) & set(val_indices.tolist())
print('Классы train:', torch.bincount(y_train).tolist())
print('Классы validation:', torch.bincount(y_val).tolist())

factories = {'MLP': make_mlp, 'CNN': make_cnn}
parameter_counts = {name: sum(p.numel() for p in factory().parameters() if p.requires_grad)
                    for name, factory in factories.items()}
print('Обучаемые параметры:', parameter_counts)
relative_gap = abs(parameter_counts['MLP'] - parameter_counts['CNN'])/parameter_counts['CNN']
print(f'Различие относительно CNN: {100*relative_gap:.2f}%')
assert relative_gap < .01


<a id="section-10"></a>

## 10. Обучение и валидация

Повторяем организацию третьего семинара: шаг обучения, обход батчей,
оценка без изменения весов. Среднюю потерю считаем по объектам, не как среднее
средних батчей разного размера. Время измеряем с синхронизацией устройства.

Все три seed составляют основной эксперимент. Один seed или одна эпоха подходят
для проверки запуска, но не подменяют итоговое сравнение.
**Вопрос:** почему loss по ходу обучения и train loss после эпохи — не одно и то же измерение?


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def synchronize():
    if device.type == 'mps':
        torch.mps.synchronize()
    elif device.type == 'cuda':
        torch.cuda.synchronize()

def train_epoch(model, loader, optimizer):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        optimizer.step()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_sum, correct, count = 0., 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss_sum += F.cross_entropy(logits, yb, reduction='sum').item()
        correct += (logits.argmax(1) == yb).sum().item()
        count += len(yb)
    return loss_sum/count, correct/count


In [ ]:
seeds = [17, 29, 43]
n_epochs = 5
batch_size = 128
lr = .001
train_eval_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
runs, trained_models = [], {}
for seed in seeds:
    for name, factory in factories.items():
        seed_everything(seed)
        model = factory().to(device)
        # Отдельный генератор: инициализация модели не меняет порядок батчей.
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                                  generator=torch.Generator().manual_seed(seed), num_workers=0)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        history = []
        synchronize()
        started = perf_counter()
        for epoch in range(n_epochs):
            train_epoch(model, train_loader, optimizer)
            train_loss, train_accuracy = evaluate(model, train_eval_loader)
            val_loss, val_accuracy = evaluate(model, val_loader)
            history.append(dict(train_loss=train_loss, train_accuracy=train_accuracy,
                                val_loss=val_loss, val_accuracy=val_accuracy))
        synchronize()
        elapsed = perf_counter() - started
        runs.append(dict(model=name, seed=seed, history=history, seconds=elapsed))
        trained_models[(name, seed)] = model.cpu()
        print(f'{name}, seed={seed}: val accuracy={val_accuracy:.3f}, время={elapsed:.1f} с')


In [ ]:
seminar_plots.plot_training(runs);
for name in factories:
    values = np.array([r['history'][-1]['val_accuracy'] for r in runs if r['model'] == name])
    print(f'{name}: среднее={values.mean():.4f}, std={values.std(ddof=1):.4f}, запуски={values}')
print('Время выше включает обучение и оценку после каждой эпохи; это не чистое время forward.')


<a id="section-11"></a>

## 11. Фильтры и карты признаков

Вес фильтра и карта признаков — разные объекты: карта зависит и от весов, и от входа.
В стиле Chapter 5 посмотрим на фильтры и выход первого блока.
Выбираем seed и объект заранее, а не по красоте картинки.

**Вопрос:** можно ли назвать канал «детектором цифры» только по одному изображению его отклика?


In [ ]:
visual_model = trained_models[('CNN', seeds[0])].eval()
with torch.no_grad():
    first_maps = visual_model[:2](x_val[:1])
seminar_plots.show_images(visual_model[0].weight[:, 0],
                         [f'Ядро {i}' for i in range(8)], signed=True);
seminar_plots.show_images(first_maps[0], [f'Карта {i}, после ReLU' for i in range(8)]);


<a id="section-12"></a>

## 12. Проверяем сдвиги

Эквивариантность: при сдвиге входа сдвигается и выход. Инвариантность:
выход не меняется. Обычная свёртка с общими весами поддерживает первое свойство
при согласованных сдвигах и граничных условиях. Stride, pooling и обрезание ограничивают его.
Это не гарантия инвариантности классификатора.

Сдвигаем все валидационные изображения на 2 и 4 пикселя в четырёх направлениях,
заполняя фон значением -1 после нормировки. Одни и те же преобразования применяем к обеим моделям.
Показываем всю выборку и отдельно общую подвыборку, где ни один из этих сдвигов
не обрезает ненулевые исходные пиксели. Она может иметь другой состав классов.

**Перед запуском:** гарантирует ли среднее по пространству устойчивость к сдвигу
на один пиксель, если раньше в сети был pooling со stride 2?


In [ ]:
def shift_images(x, dy, dx, fill=-1.):
    out = torch.full_like(x, fill)
    h, w = x.shape[-2:]
    if abs(dy) >= h or abs(dx) >= w:
        return out
    sy, sx = max(0, -dy), max(0, -dx)
    ty, tx = max(0, dy), max(0, dx)
    height, width = h-abs(dy), w-abs(dx)
    out[..., ty:ty+height, tx:tx+width] = x[..., sy:sy+height, sx:sx+width]
    return out

shifts = [(0, 0)] + [(dy, dx) for d in [2, 4] for dy, dx in [(0,d), (0,-d), (d,0), (-d,0)]]
foreground = x_val > -1
border = foreground.clone()
border[..., 4:-4, 4:-4] = False
interior_mask = ~border.flatten(1).any(1)
print('Общая подвыборка без обрезки:', int(interior_mask.sum()), 'из', len(x_val))
print('Её состав классов:', torch.bincount(y_val[interior_mask], minlength=10).tolist())
assert interior_mask.any()
seminar_plots.show_images([shift_images(x_val[:1], dy, dx) for dy, dx in shifts[:5]],
                         [f'dy={dy}, dx={dx}' for dy, dx in shifts[:5]]);


In [ ]:
@torch.no_grad()
def predict_classes(model, inputs):
    model.eval()
    return torch.cat([model(batch.to(device)).argmax(1).cpu()
                      for batch in inputs.split(batch_size)])

shift_results = []
for (name, seed), model in trained_models.items():
    model.to(device)
    for dy, dx in shifts:
        predicted = predict_classes(model, shift_images(x_val, dy, dx))
        for subset, mask in [('all', torch.ones(len(y_val), dtype=torch.bool)), ('interior', interior_mask)]:
            accuracy = (predicted[mask] == y_val[mask]).float().mean().item()
            shift_results.append(dict(model=name, seed=seed, dy=dy, dx=dx,
                                      distance=max(abs(dy),abs(dx)), subset=subset,
                                      n=int(mask.sum()), accuracy=accuracy))
    model.cpu()
seminar_plots.plot_shifts(shift_results);
print('На графике: тонкие линии — seed; толстая — среднее; при d>0 усреднены четыре направления.')


In [ ]:
for subset in ['all', 'interior']:
    print('Выборка:', subset)
    for name in factories:
        drops = []
        for seed in seeds:
            rows = [r for r in shift_results if r['model']==name and r['seed']==seed and r['subset']==subset]
            base = next(r['accuracy'] for r in rows if r['distance']==0)
            moved = np.mean([r['accuracy'] for r in rows if r['distance']==4])
            drops.append(100*(base-moved))
        print(name, 'падение accuracy при сдвиге 4, п.п.:', np.round(drops, 2))


<a id="section-13"></a>

## 13. Задания

Задания независимы. Сначала сформулируйте прогноз или запишите решение,
затем откройте проверку. Дополнительное обучение не требуется.


### T1. Форма выхода

Для входа `(2, 3, 15, 17)` слой Conv2d(3, 4, 3, stride=2, padding=1) выдаст какую форму? Проверьте число обучаемых параметров.


In [ ]:
# YOUR CODE HERE


<details>
<summary>Ответ и проверка T1</summary>

```python
layer = nn.Conv2d(3, 4, 3, stride=2, padding=1)
assert layer(torch.zeros(2, 3, 15, 17)).shape == (2, 4, 8, 9)
assert sum(p.numel() for p in layer.parameters()) == 112
```

</details>


### T2. Корреляция или свёртка

Для сигнала [1, 2, 4, 8] и ядра [1, -1] посчитайте F.conv1d без padding. Что изменится при перевороте ядра?


In [ ]:
# YOUR CODE HERE


<details>
<summary>Ответ и проверка T2</summary>

```python
x = torch.tensor([[[1., 2., 4., 8.]]])
k = torch.tensor([[[1., -1.]]])
torch.testing.assert_close(F.conv1d(x, k), torch.tensor([[[-1., -2., -4.]]]))
torch.testing.assert_close(F.conv1d(x, k.flip(-1)), -F.conv1d(x, k))
```

</details>


### T3. Каналы 1x1

Постройте слой без bias, который превращает три канала в один, усредняя их. Не усредняйте пространственные позиции.


In [ ]:
# YOUR CODE HERE


<details>
<summary>Ответ и проверка T3</summary>

```python
layer = nn.Conv2d(3, 1, 1, bias=False)
with torch.no_grad():
    layer.weight.fill_(1/3)
x = torch.arange(24.).view(1, 3, 2, 4)
torch.testing.assert_close(layer(x), x.mean(1, keepdim=True))
```

</details>


### T4. Собель

Компоненты градиента равны 3 и -4. Чему равны сумма и длина? Проверьте вычисления тензорами.


In [ ]:
# YOUR CODE HERE


<details>
<summary>Ответ и проверка T4</summary>

```python
g = torch.tensor([3., -4.])
assert g.sum().item() == -1
assert g.square().sum().sqrt().item() == 5
```

</details>


### T5. Эквивариантность

Для случайного входа 9x9 и ядра 3x3 проверьте сдвиг на один пиксель вправо. Сравнивайте только область без влияния границ. Используйте stride=1, padding=0.


In [ ]:
# YOUR CODE HERE


<details>
<summary>Ответ и проверка T5</summary>

```python
g = torch.Generator().manual_seed(11)
x = torch.randn(1, 1, 9, 9, generator=g)
k = torch.randn(1, 1, 3, 3, generator=g)
shifted = torch.zeros_like(x)
shifted[..., 1:] = x[..., :-1]
original = F.conv2d(x, k)
moved = F.conv2d(shifted, k)
torch.testing.assert_close(moved[..., 1:], original[..., :-1])
```

</details>


### T6. Конструктивное сравнение

У CNN accuracy 0.94, у MLP 0.93 на одном seed. Достаточно ли этого для вывода об общем превосходстве CNN? Назовите три проверки и посчитайте разность в процентных пунктах.


In [ ]:
# YOUR CODE HERE


<details>
<summary>Ответ и проверка T6</summary>

```python
difference_pp = 100*(.94-.93)
assert abs(difference_pp-1) < 1e-10
# Нужны повторные запуски, общий протокол и оценка ограничений сравнения.
# Даже после них вывод относится к этим моделям и этим данным.
```

</details>


<a id="section-14"></a>

## 14. Итоги

Свёртка задаёт локальную связность и общие веса. Pooling и глобальное
усреднение меняют способ агрегирования признаков. Это полезные предположения о данных,
но не доказательство, что любая CNN всегда лучше любой MLP.

**Обсуждение:** что показали наши запуски? Насколько результат зависит от seed,
сдвига и обрезания? Какой следующий эксперимент отличит эффект общих весов
от эффекта pooling и усреднения?

Следующий шаг — архитектуры CNN и организация блоков. Для ДЗ1 остаются прежние
полносвязная архитектура и условия; этот семинар их не меняет.


## Источники

- D. V. Godoy: [Chapter 4](https://github.com/dvgodoy/PyTorchStepByStep/blob/master/Chapter04.ipynb),
  [Chapter 5](https://github.com/dvgodoy/PyTorchStepByStep/blob/master/Chapter05.ipynb).
- Числовой пример фильтра и последовательность базовых операций адаптированы из Chapter 5,
  MIT, copyright (c) 2020 Daniel Voigt Godoy; текст лицензии в корне репозитория: `LICENSE-GODOY`.
- Демонстрационное фото и идеи упражнений: семинар 4 курса ML3 2025.
- PyTorch: [Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html).

Обучение и сопоставление MNIST в этом notebook — дополнение курса, а не воспроизведение
численных результатов Годоя. StepByStep и собственный backward не требуются.
